In [3]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

# ---- same configuration as the main activity ----
LAST_NAME  = "Agawa"
STUDENT_ID = "TUPM-26-1082"
id_digits  = "".join(d for d in STUDENT_ID if d.isdigit())

SEED_DIGIT  = int(id_digits[-1])
ID_SUM      = sum(map(int, id_digits))
RANDOM_SEED = int(id_digits) % (2**32)
# --------------------------------------------------

# Original SECOM files stay in the MAIN activity's data folder (not copied here)
candidates = [
    Path(r"C:\Users\Agawa\Downloads\secom"),
    Path.cwd() / "data", Path.cwd().parent / "data", Path.cwd().parent / "Lab7_Numpy_Pandas" / "data",
]
DATA_DIR = next((p for p in candidates if (p / "secom.data").is_file()), None)
assert DATA_DIR is not None, "secom.data not found - edit DATA_DIR above"

warnings.filterwarnings("ignore", category=RuntimeWarning)

raw = np.genfromtxt(DATA_DIR / "secom.data", dtype=float)
y   = np.loadtxt(DATA_DIR / "secom_labels.data", usecols=0, dtype=int)
assert raw.shape[0] == len(y) and np.isin(y, [-1, 1]).all()

X_df = pd.DataFrame(raw, columns=[f"Feature_{i:03d}" for i in range(raw.shape[1])])
X_df.index.name = "Example_ID"                      # original record identifier
outcome = np.where(y == -1, "Pass", "Fail")

print("Seed Digit:", SEED_DIGIT, "| ID Sum:", ID_SUM, "| Random Seed:", RANDOM_SEED)
print("Matrix:", X_df.shape, "| Pass:", (y == -1).sum(), "| Fail:", (y == 1).sum())

Seed Digit: 2 | ID Sum: 19 | Random Seed: 261082
Matrix: (1567, 590) | Pass: 1463 | Fail: 104


In [4]:
X = X_df.to_numpy(dtype=float)
n_rows, n_feat = X.shape
names = np.array(X_df.columns)

miss_mask  = np.isnan(X)
miss_count = miss_mask.sum(axis=0)
n_obs      = n_rows - miss_count

fmin = np.nanmin(X, axis=0)
fmax = np.nanmax(X, axis=0)

has_missing = miss_count > 0
is_constant = (fmin == fmax)              # all observed values identical
all_missing = n_obs == 0
usable      = ~(is_constant | all_missing)

print("Features with missing values :", has_missing.sum())
print("Constant features            :", is_constant.sum())
print("All-missing features         :", all_missing.sum())
print("Usable features              :", usable.sum())

Features with missing values : 538
Constant features            : 116
All-missing features         : 0
Usable features              : 474


In [5]:
k1 = 5 + (SEED_DIGIT % 3)
rng = np.random.default_rng(RANDOM_SEED)
sel1 = np.sort(rng.choice(np.flatnonzero(usable), size=k1, replace=False))
Xs = X[:, sel1]
sel1_names = names[sel1]
print("Selected features:", list(sel1_names))

Selected features: ['Feature_078', 'Feature_268', 'Feature_272', 'Feature_302', 'Feature_558', 'Feature_564', 'Feature_574']


In [7]:
means   = np.nanmean(Xs, axis=0)
medians = np.nanmedian(Xs, axis=0)
stds    = np.nanstd(Xs, axis=0, ddof=1)          # sample std
miss_pct = np.isnan(Xs).mean(axis=0) * 100
n_obs_s  = (~np.isnan(Xs)).sum(axis=0)

In [8]:
mad   = np.nanmedian(np.abs(Xs - medians), axis=0)
scale = 1.4826 * mad                              # MAD scaled to match std
ok    = scale > 0                                 # zero-MAD features cannot be scaled

robust_z = np.full(Xs.shape, np.nan)
np.divide(Xs - medians, scale, out=robust_z, where=np.broadcast_to(ok, Xs.shape))
print("Features with zero MAD (left unflagged):", list(sel1_names[~ok]))

Features with zero MAD (left unflagged): []


In [9]:
flags = np.abs(robust_z) > 3.5                    # NaN compares False
flag_count = flags.sum(axis=0)
flag_pct   = flag_count / n_obs_s * 100
total_flagged = int(flag_count.sum())
print("Flagged per feature:", dict(zip(sel1_names, flag_count)))

Flagged per feature: {'Feature_078': np.int64(2), 'Feature_268': np.int64(0), 'Feature_272': np.int64(1), 'Feature_302': np.int64(8), 'Feature_558': np.int64(120), 'Feature_564': np.int64(109), 'Feature_574': np.int64(85)}


In [10]:
summary1 = pd.DataFrame({
    "feature": sel1_names, "mean": means, "median": medians, "std_sample": stds,
    "missing_pct": miss_pct, "mad": mad, "flagged_count": flag_count, "flagged_pct": flag_pct,
}).sort_values("flagged_pct", ascending=False).reset_index(drop=True)
summary1.insert(0, "rank", summary1.index + 1)
summary1.to_csv("measurement_quality.csv", index=False)
summary1

,rank,feature,mean,median,std_sample,missing_pct,mad,flagged_count,flagged_pct
0,1,Feature_564,6.444985,5.16000,2.633583,17.421825,1.0600,109,8.423493
1,2,Feature_558,0.995009,0.97270,0.083860,0.063816,0.0209,120,7.662835
2,3,Feature_574,9.162315,2.97580,26.920150,0.000000,0.4514,85,5.424378
3,4,Feature_302,0.403342,0.38080,0.197514,0.127632,0.0894,8,0.511182
4,5,Feature_078,-0.013643,-0.01250,0.047872,1.531589,0.0302,2,0.129618
5,6,Feature_272,41.298147,40.01925,17.737513,0.446713,13.6238,1,0.064103
6,7,Feature_268,19.504677,17.97700,7.344404,0.510530,5.1980,0,0.000000


In [11]:
chk = X_df[list(sel1_names)]
v_mean   = np.allclose(chk.mean().to_numpy(), means)
v_median = np.allclose(chk.median().to_numpy(), medians)
v_std    = np.allclose(chk.std(ddof=1).to_numpy(), stds)
v_miss   = np.allclose(chk.isna().mean().to_numpy() * 100, miss_pct)
pd_mad   = (chk - chk.median()).abs().median()
v_mad    = np.allclose(pd_mad.to_numpy(), mad)
pd_flags = ((chk - chk.median()).div(1.4826 * pd_mad.where(pd_mad > 0))).abs() > 3.5
v_flags  = int(pd_flags.sum().sum()) == total_flagged
verified1 = all([v_mean, v_median, v_std, v_miss, v_mad, v_flags])

print("\n--- ASSESSMENT DATA: EXERCISE 1 ---")
print("Selected Features                  :", list(sel1_names))
print("Highest Missing Percentage         : %.4f%% (%s)" % (miss_pct.max(), sel1_names[miss_pct.argmax()]))
print("Feature with Highest Flagged %%     : %s (%.4f%%)" % (summary1.loc[0, "feature"], summary1.loc[0, "flagged_pct"]))
print("Total Flagged Observations         :", total_flagged)
print("Vectorized Results Verified        :", verified1)


--- ASSESSMENT DATA: EXERCISE 1 ---
Selected Features                  : ['Feature_078', 'Feature_268', 'Feature_272', 'Feature_302', 'Feature_558', 'Feature_564', 'Feature_574']
Highest Missing Percentage         : 17.4218% (Feature_564)
Feature with Highest Flagged %     : Feature_564 (8.4235%)
Total Flagged Observations         : 325
Vectorized Results Verified        : True


In [12]:
df = X_df.copy()
df["Outcome"] = outcome
n_pass = (df["Outcome"] == "Pass").sum()
n_fail = (df["Outcome"] == "Fail").sum()
print(n_pass, n_fail)
df[["Outcome"]].head()

1463 104


,Outcome
Example_ID,
0,Pass
1,Pass
2,Fail
3,Pass
4,Pass


In [13]:
n_per = int(min(20 + ID_SUM, n_pass, n_fail))
balanced = df.groupby("Outcome", group_keys=False).sample(n=n_per, random_state=RANDOM_SEED)
balanced_size = len(balanced)
print("Per outcome:", n_per, "| Balanced sample size:", balanced_size)

Per outcome: 39 | Balanced sample size: 78


In [14]:
rng2 = np.random.default_rng(RANDOM_SEED)
sel2 = list(names[np.sort(rng2.choice(np.flatnonzero(usable), size=5, replace=False))])
full_medians = X_df[sel2].median()

bal = balanced[sel2 + ["Outcome"]].copy()
bal[sel2] = bal[sel2].fillna(full_medians)
bal.to_csv("balanced_sample.csv")                 # keeps record_id
print("Selected Features:", sel2)

Selected Features: ['Feature_079', 'Feature_269', 'Feature_273', 'Feature_560', 'Feature_566']


In [15]:
agg = bal.groupby("Outcome")[sel2].agg(["count", "mean", "median", "std"])   # std = sample std (ddof=1)
agg

Feature_079                             Feature_269                    \
              count      mean  median       std       count      mean  median   
Outcome                                                                         
Fail             39  0.007272  0.0059  0.022519          39  3.722459  3.9099   
Pass             39  0.004795  0.0000  0.020608          39  3.740195  3.7035   

                  Feature_273                               Feature_560  \
              std       count       mean   median       std       count   
Outcome                                                                   
Fail     1.083875          39  20.090569  18.9326  3.736833          39   
Pass     0.956603          39  20.120569  19.6525  3.054363          39   

                                    Feature_566                               
             mean  median       std       count      mean   median       std  
Outcome                                                                       
Fail     0.079677  0.0722  0.046456          39  2.359224  2.15045  0.800177  
Pass     0.068321  0.0488  0.043942          39  2.883541  2.15045  1.234620

In [16]:
rows = []
for f in sel2:
    g = bal.groupby("Outcome")[f].agg(["count", "mean", "median", "std"])
    n1, n2 = g.loc["Pass", "count"], g.loc["Fail", "count"]
    s1, s2 = g.loc["Pass", "std"],   g.loc["Fail", "std"]
    pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    abs_diff = abs(g.loc["Fail", "mean"] - g.loc["Pass", "mean"])
    smd = abs_diff / pooled if pooled > 0 else np.nan          # zero denominator -> NaN
    rows.append({
        "feature": f,
        "pass_count": n1, "fail_count": n2,
        "pass_mean": g.loc["Pass", "mean"], "fail_mean": g.loc["Fail", "mean"],
        "pass_median": g.loc["Pass", "median"], "fail_median": g.loc["Fail", "median"],
        "pass_std": s1, "fail_std": s2,
        "pooled_std": pooled, "abs_mean_diff": abs_diff, "abs_std_mean_diff": smd,
    })
comp = pd.DataFrame(rows)

In [17]:
comp = comp.sort_values("abs_std_mean_diff", ascending=False, na_position="last").reset_index(drop=True)
comp.insert(0, "rank", comp.index + 1)
comp.to_csv("feature_comparison.csv", index=False)
comp[["rank", "feature", "abs_mean_diff", "abs_std_mean_diff"]]

,rank,feature,abs_mean_diff,abs_std_mean_diff
0,1,Feature_566,0.524317,0.503991
1,2,Feature_560,0.011356,0.251157
2,3,Feature_079,0.002477,0.114754
3,4,Feature_269,0.017736,0.017350
4,5,Feature_273,0.030000,0.008791


In [18]:
# NumPy vs Pandas statistics match
m_ok = True
for f in sel2:
    for lab in ["Pass", "Fail"]:
        vals = bal.loc[bal["Outcome"] == lab, f].to_numpy()
        r = comp.loc[comp["feature"] == f].iloc[0]
        m_ok &= np.isclose(vals.mean(), r[f"{lab.lower()}_mean"])
        m_ok &= np.isclose(np.median(vals), r[f"{lab.lower()}_median"])
        m_ok &= np.isclose(vals.std(ddof=1), r[f"{lab.lower()}_std"])

print("\n--- ASSESSMENT DATA: EXERCISE 2 ---")
print("Balanced Sample Size               :", balanced_size, f"({n_per} Pass + {n_per} Fail)")
print("Selected Features                  :", sel2)
print("Highest-Ranked Feature             :", comp.loc[0, "feature"])
print("Absolute Standardized Mean Diff    : %.6f" % comp.loc[0, "abs_std_mean_diff"])
print("NumPy-Pandas Statistics Match      :", bool(m_ok))


--- ASSESSMENT DATA: EXERCISE 2 ---
Balanced Sample Size               : 78 (39 Pass + 39 Fail)
Selected Features                  : ['Feature_079', 'Feature_269', 'Feature_273', 'Feature_560', 'Feature_566']
Highest-Ranked Feature             : Feature_566
Absolute Standardized Mean Diff    : 0.503991
NumPy-Pandas Statistics Match      : True


In [19]:
is_fail = (outcome == "Fail")
obs_pass = (~np.isnan(X[~is_fail])).sum(axis=0)
obs_fail = (~np.isnan(X[is_fail])).sum(axis=0)
eligible = usable & (obs_pass >= 10) & (obs_fail >= 10)

rng3 = np.random.default_rng(RANDOM_SEED)
sel3 = np.sort(rng3.choice(np.flatnonzero(eligible), size=5, replace=False))
sel3_names = list(names[sel3])
print("Eligible features:", eligible.sum(), "| Selected:", sel3_names)

XP = X[~is_fail][:, sel3]       # Pass rows
XF = X[is_fail][:, sel3]        # Fail rows

Eligible features: 470 | Selected: ['Feature_078', 'Feature_269', 'Feature_273', 'Feature_560', 'Feature_567']


In [20]:
med_full = np.nanmedian(X[:, sel3], axis=0)

def impute(M):
    return np.where(np.isnan(M), med_full, M)

XP_B, XF_B = impute(XP), impute(XF)

# Fail - Pass mean difference
diff_A = np.nanmean(XF, axis=0)   - np.nanmean(XP, axis=0)
diff_B = np.nanmean(XF_B, axis=0) - np.nanmean(XP_B, axis=0)
print("Diff A:", diff_A)
print("Diff B:", diff_B)

Diff A: [-0.00830466  0.08325143  0.37848036  0.00415255  0.00624201]
Diff B: [-0.00831423  0.08362771  0.38398275  0.00416141  0.0041322 ]


In [21]:
N_BOOT = 200 + 10 * SEED_DIGIT

def bootstrap(seed):
    rng = np.random.default_rng(seed)
    iP = rng.integers(0, XP.shape[0], size=(N_BOOT, XP.shape[0]))   # Pass indices
    iF = rng.integers(0, XF.shape[0], size=(N_BOOT, XF.shape[0]))   # Fail indices
    bA = np.nanmean(XF[iF],   axis=1) - np.nanmean(XP[iP],   axis=1)    # strategy A
    bB = np.nanmean(XF_B[iF], axis=1) - np.nanmean(XP_B[iP], axis=1)    # strategy B
    return bA, bB

bA, bB = bootstrap(RANDOM_SEED)
print("Bootstrap repetitions:", N_BOOT)

Bootstrap repetitions: 220


In [22]:
def summarize(b):
    return np.nanmean(b, axis=0), np.nanpercentile(b, 2.5, axis=0), np.nanpercentile(b, 97.5, axis=0)

bmA, loA, hiA = summarize(bA)
bmB, loB, hiB = summarize(bB)

def pooled_std(P, F):
    n1, n2 = (~np.isnan(P)).sum(0), (~np.isnan(F)).sum(0)
    s1, s2 = np.nanstd(P, axis=0, ddof=1), np.nanstd(F, axis=0, ddof=1)
    return np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))

def abs_smd(diff, ps):
    return np.divide(np.abs(diff), ps, out=np.full_like(diff, np.nan), where=ps > 0)

smdA = abs_smd(diff_A, pooled_std(XP, XF))
smdB = abs_smd(diff_B, pooled_std(XP_B, XF_B))

def rank_desc(v):                      # 1 = largest; NaN ranked last
    v = np.where(np.isnan(v), -np.inf, v)
    return (-v).argsort().argsort() + 1

rankA, rankB = rank_desc(smdA), rank_desc(smdB)

In [23]:
sign_changed   = np.sign(diff_A) != np.sign(diff_B)
rank_changed   = rankA != rankB
ci_excl0_A     = (loA > 0) | (hiA < 0)           # CI excludes zero?
ci_excl0_B     = (loB > 0) | (hiB < 0)
change         = np.abs(diff_B - diff_A)

robust = pd.DataFrame({
    "feature": sel3_names,
    "diff_A": diff_A, "boot_mean_A": bmA, "ci_low_A": loA, "ci_high_A": hiA, "rank_A": rankA,
    "diff_B": diff_B, "boot_mean_B": bmB, "ci_low_B": loB, "ci_high_B": hiB, "rank_B": rankB,
    "sign_changed": sign_changed, "rank_changed": rank_changed,
    "CI_excludes_0_A": ci_excl0_A, "CI_excludes_0_B": ci_excl0_B,
    "abs_change_in_mean_diff": change,
})
robust.to_csv("robustness_comparison.csv", index=False)
robust

,feature,diff_A,boot_mean_A,ci_low_A,ci_high_A,rank_A,diff_B,boot_mean_B,ci_low_B,ci_high_B,rank_B,sign_changed,rank_changed,CI_excludes_0_A,CI_excludes_0_B,abs_change_in_mean_diff
0,Feature_078,-0.008305,-0.008033,-0.016648,0.002801,2,-0.008314,-0.008044,-0.016641,0.002795,1,False,True,False,False,0.000010
1,Feature_269,0.083251,0.071863,-0.154088,0.331032,5,0.083628,0.072274,-0.153771,0.331326,5,False,False,False,False,0.000376
2,Feature_273,0.378480,0.381529,-0.315916,1.032591,3,0.383983,0.387083,-0.310089,1.040893,3,False,False,False,False,0.005502
3,Feature_560,0.004153,0.004503,-0.004298,0.012758,4,0.004161,0.004513,-0.004293,0.012770,4,False,False,False,False,0.000009
4,Feature_567,0.006242,0.006307,-0.001649,0.016110,1,0.004132,0.004241,-0.002136,0.012175,2,False,True,False,False,0.002110


In [24]:
bA2, bB2 = bootstrap(RANDOM_SEED)
reproducible = bool(np.array_equal(bA, bA2) and np.array_equal(bB, bB2))
print("Reproducible:", reproducible)

Reproducible: True


In [25]:
imax = int(change.argmax())
lines = [
    "SENSITIVITY REPORT - SECOM (Fail minus Pass mean difference)",
    f"Seed: {RANDOM_SEED} | SEED_DIGIT: {SEED_DIGIT} | Bootstrap repetitions: {N_BOOT}",
    f"Selected features: {', '.join(sel3_names)}",
    "Strategy A = available-case. Strategy B = median imputation (full-dataset medians).",
    "",
    f"Features with changed difference sign: {int(sign_changed.sum())}",
    f"Features with changed ranking       : {int(rank_changed.sum())}",
    f"Largest change in mean difference   : {change[imax]:.6f} ({sel3_names[imax]})",
    f"Features whose 95% CI excludes zero  : A = {int(ci_excl0_A.sum())}, B = {int(ci_excl0_B.sum())}",
    f"Reproducibility verified            : {reproducible}",
    "",
    "Interpretation:",
]
if sign_changed.sum() == 0 and rank_changed.sum() == 0:
    lines.append("Direction and ranking were identical under both strategies, so the conclusions are stable to the preprocessing choice.")
else:
    lines.append("Some features changed direction or ranking between strategies, so conclusions for those features depend on the preprocessing method and should be treated with caution.")
lines.append("Features whose CI includes zero do not show a reliable Fail-vs-Pass difference. Imputing with the median pulls values toward the center, which can shrink the spread and the difference.")

with open("sensitivity_report.txt", "w") as f:
    f.write("\n".join(lines))
print("\n".join(lines))

SENSITIVITY REPORT - SECOM (Fail minus Pass mean difference)
Seed: 261082 | SEED_DIGIT: 2 | Bootstrap repetitions: 220
Selected features: Feature_078, Feature_269, Feature_273, Feature_560, Feature_567
Strategy A = available-case. Strategy B = median imputation (full-dataset medians).

Features with changed difference sign: 0
Features with changed ranking       : 2
Largest change in mean difference   : 0.005502 (Feature_273)
Features whose 95% CI excludes zero  : A = 0, B = 0
Reproducibility verified            : True

Interpretation:
Some features changed direction or ranking between strategies, so conclusions for those features depend on the preprocessing method and should be treated with caution.
Features whose CI includes zero do not show a reliable Fail-vs-Pass difference. Imputing with the median pulls values toward the center, which can shrink the spread and the difference.


In [26]:
print("\n--- ASSESSMENT DATA: EXERCISE 3 ---")
print("Selected Features                  :", sel3_names)
print("Bootstrap Repetitions              :", N_BOOT)
print("Largest Change in Mean Difference  : %.6f (%s)" % (change[imax], sel3_names[imax]))
print("Features with Changed Diff. Sign   :", int(sign_changed.sum()))
print("Features with Changed Ranking      :", int(rank_changed.sum()))
print("Reproducibility Verified           :", reproducible)


--- ASSESSMENT DATA: EXERCISE 3 ---
Selected Features                  : ['Feature_078', 'Feature_269', 'Feature_273', 'Feature_560', 'Feature_567']
Bootstrap Repetitions              : 220
Largest Change in Mean Difference  : 0.005502 (Feature_273)
Features with Changed Diff. Sign   : 0
Features with Changed Ranking      : 2
Reproducibility Verified           : True
